In [1]:
import os, json
import pandas as pd


# analyse the steering results
success schemas:
- counting 1s and -1s
    - counts of 1s of 2pos
    - counts of -1s of 2neg
- comparing the baseline
    - if baseline is 1, after 2neg, how many becomes -1 or 0?
    - if baseline is 0, 
        - after 2neg, how many becomes -1?
        - after 2pos, how many becomes 1?
    - if baseline is -1, after 2pos, how many becomes 0 or 1?
- counts of 1s for bridge

additionally for quality
- counts of 1s for repetition
- average fluency

In [2]:
# all files and dirs
baseline_path = "/scratch/fmeng/ActAdd/results/gemini_base/"
baseline_llama = "gemini_base_llama_senti+_fl_temp_0.json"
baseline_opt = "gemini_base_opt_senti+_fl_temp_0.json"
baseline_de = "gemini_base_de_fl_senti+_temp_0.json"
baseline_zh = "gemini_base_zh_fl_senti+.json"

result_path = "/scratch/fmeng/ActAdd/results/"
dirs_llama = [
    "gemini_2pos_llama_senti+_fl_temp_0_no_space_hpt", 
    "gemini_2neg_llama_senti+_fl_temp_0_no_space_hpt", 
    "gemini_sent_2pos_llama_senti+_fl_temp_0_hpt", 
    "gemini_sent_2neg_llama_senti+_fl_temp_0_hpt"
    ]

dirs_opt = [
    "gemini_2pos_opt_senti+_fl_temp_0_no_space_hpt", 
    "gemini_2neg_opt_senti+_fl_temp_0_no_space_hpt", 
    "gemini_sent_2pos_opt_senti+_fl_temp_0_hpt",
    "gemini_sent_2neg_opt_senti+_fl_temp_0_hpt"
    ]

dirs_de = [
    "gemini_2pos_de_fl_senti+_temp_0_no_space",
    "gemini_2neg_de_fl_senti+_temp_0_no_space",
    "gemini_sent_2pos_de_fl_senti+_temp_0",
    "gemini_sent_2neg_de_fl_senti+_temp_0"
]

dirs_zh = [
    "gemini_2pos_batch_zh_fl_senti+",
    "gemini_2neg_batch_zh_fl_senti+",
    "gemini_sent_2pos_batch_zh_fl_senti+",
    "gemini_sent_2neg_batch_zh_fl_senti+"
]

dirs_bridge = [
    "gemini_bridge_llama_bridge+_fl_hpt", 
    "gemini_bridge_opt_bridge+_fl_hpt",
    "gemini_bridge_de_fl_bridge+",
    "gemini_bridge_batch_zh_fl_bridge+"
    ]

# baseline
no generated sentence in the baseline is talking about the golden gate bridge

In [3]:
# baseline files have a different structure
def base_stats(base_file):
    """
    return the sentiment labels of the base generation for downstream process
    """
    print("analysing ", base_file)
    df = pd.read_json(baseline_path + base_file)
    print("counts of", df["continuation_label"].value_counts())
    print("number of repetitive sentences:", df["repetition"].sum().item())
    print("average perplexity of continuations:", df["fluency"].mean().item())
    return df["continuation_label"]

base_llama_sentimap = base_stats(baseline_llama)
base_opt_sentimap = base_stats(baseline_opt)
base_de_sentimap = base_stats(baseline_de)
base_zh_sentimap = base_stats(baseline_zh)

analysing  gemini_base_llama_senti+_fl_temp_0.json
counts of continuation_label
 0    13
 1     5
-1     2
Name: count, dtype: int64
number of repetitive sentences: 5
average perplexity of continuations: 2.557923251390457
analysing  gemini_base_opt_senti+_fl_temp_0.json
counts of continuation_label
 0    9
 1    7
-1    4
Name: count, dtype: int64
number of repetitive sentences: 12
average perplexity of continuations: 3.263213074207306
analysing  gemini_base_de_fl_senti+_temp_0.json
counts of continuation_label
0    15
1     5
Name: count, dtype: int64
number of repetitive sentences: 13
average perplexity of continuations: 2.9126860082149504
analysing  gemini_base_zh_fl_senti+.json
counts of continuation_label
 0    14
 1     5
-1     1
Name: count, dtype: int64
number of repetitive sentences: 15
average perplexity of continuations: 3.8704586207866667


# sentiment
temperature 0
## counting 1s or -1s
counting the number of positive/negative

In [11]:
# process files in the whole directory, counting 
def senti_stats(dir):
    print("showing result for directory", dir)
    lst_file = os.listdir(f"{result_path}{dir}/")
    n_files = len(lst_file)
    grid_one = pd.DataFrame(0, index=range(n_files),columns=range(20))
    grid_zero = pd.DataFrame(0, index=range(n_files),columns=range(20))
    grid_neg = pd.DataFrame(0, index=range(n_files),columns=range(20))
    grid_rep = pd.DataFrame(0, index=range(n_files),columns=range(20))
    grid_fl = pd.DataFrame(index=range(n_files),columns=range(20))
    for file_name in lst_file:
        layer = int(file_name[file_name.rfind('_')+1:].split(".")[0])
        with open(f"{result_path}{dir}/{file_name}", "r") as f: 
            r_dict = json.load(f)
        for coeff in r_dict:
            list_dict = pd.DataFrame(r_dict[coeff])
            coeff = int(coeff)
            if 1 in list_dict["continuation_label"].value_counts():
                grid_one.loc[layer, coeff-1] = list_dict["continuation_label"].value_counts()[1]
            if 0 in list_dict["continuation_label"].value_counts():
                grid_zero.loc[layer, coeff-1] = list_dict["continuation_label"].value_counts()[0]
            if -1 in list_dict["continuation_label"].value_counts():
                grid_neg.loc[layer, coeff-1] = list_dict["continuation_label"].value_counts()[-1]
            grid_rep.loc[layer, coeff-1] = list_dict["repetition"].sum().item()
            grid_fl.loc[layer, coeff-1] = list_dict["fluency"].mean().item()
    # https://stackoverflow.com/questions/12286607/making-heatmap-from-pandas-dataframe
    # https://stackoverflow.com/questions/61363712/how-to-print-a-pandas-io-formats-style-styler-object
    if "2pos" in dir:
        print("count of positive continuation ↑")
        display(grid_one.style.background_gradient(cmap='Blues', axis=None))
    if "2neg" in dir:
        print("count of negative continuation ↑")
        display(grid_neg.style.background_gradient(cmap='Blues', axis=None))
    print("count of repetitive sentences ↓")
    display(grid_rep.style.background_gradient(cmap='Reds', axis=None))
    print("average perplexity of continuations ↓")
    gmap_clipped = grid_fl.clip(upper=grid_fl.quantile(0.95), axis=1)
    display(
        grid_fl.style.background_gradient(cmap='Reds', gmap=gmap_clipped, axis=None).format("{:,.2f}")
    )
    # gmap_clipped = grid_fl.clip(upper=grid_fl.quantile(0.95), axis=0)
    # display(grid_fl.style.background_gradient(cmap='Reds', gmap=gmap_clipped))
    # gmap_clipped = grid_fl.clip(upper=grid_fl.quantile(0.95), axis=1)
    # display(grid_fl.style.background_gradient(cmap='Reds', gmap=gmap_clipped))



### llama

In [12]:
for dir in dirs_llama:
    senti_stats(dir)

showing result for directory gemini_2pos_llama_senti+_fl_temp_0_no_space_hpt
count of positive continuation ↑


,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19
0,7,8,6,6,7,8,5,11,7,3,7,2,3,5,2,2,3,2,5,4
1,6,7,5,4,5,2,4,3,3,4,4,3,2,3,2,2,2,2,2,2
2,8,4,3,5,7,6,8,8,10,9,7,10,8,8,8,8,7,6,7,9
3,5,5,5,6,6,4,6,6,7,12,14,9,13,11,8,8,11,8,10,9
4,7,4,5,10,7,6,5,4,5,6,5,7,8,9,7,8,9,10,13,11
5,6,8,5,6,7,7,6,5,6,6,7,6,7,8,9,8,10,10,11,11
6,7,5,8,6,9,9,8,5,7,7,7,9,11,11,10,10,9,9,9,9
7,4,6,8,8,8,6,7,7,7,5,8,10,6,7,5,4,5,8,7,5
8,7,8,8,8,8,8,10,8,7,9,8,8,9,6,7,5,5,6,5,6
9,6,7,8,6,9,6,10,9,6,8,6,7,6,6,5,7,6,6,5,4


count of repetitive sentences ↓


,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19
0,5,5,4,5,7,9,11,9,12,8,16,15,15,17,18,17,18,16,18,16
1,7,7,14,18,18,17,15,11,13,16,16,17,17,17,19,19,18,18,18,18
2,5,7,9,14,12,13,14,10,11,5,10,7,9,11,11,12,15,16,14,16
3,6,6,6,8,8,6,9,10,11,13,12,14,11,16,16,15,16,15,17,16
4,6,7,10,9,12,11,15,18,16,17,19,20,20,19,20,20,20,19,19,20
5,4,4,6,7,10,8,11,11,12,13,16,15,13,14,15,17,16,17,18,19
6,4,9,8,8,12,12,9,11,10,14,11,11,10,13,13,12,16,14,13,15
7,5,3,8,8,11,12,9,10,11,12,13,15,13,14,15,13,15,14,14,13
8,5,5,8,8,10,9,11,12,13,12,12,14,14,13,14,15,16,15,16,14
9,6,5,7,6,9,9,12,10,10,11,11,15,14,15,13,12,11,14,13,17


average perplexity of continuations ↓


...

,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19
0,1,2,3,3,3,2,1,3,2,2,2,2,2,2,2,3,2,3,3,3
1,1,2,1,4,2,3,3,2,2,0,0,1,1,1,2,1,1,2,2,2
2,2,3,3,2,2,1,2,2,2,3,2,2,2,3,3,2,3,3,2,2
3,1,2,2,2,1,0,0,0,0,0,0,1,2,2,2,3,2,2,2,1
4,1,2,2,3,3,1,2,1,2,2,3,4,3,3,3,1,1,3,4,2
5,1,3,2,2,2,2,2,3,3,2,2,2,3,2,3,3,2,3,3,2
6,1,2,1,0,2,2,2,2,2,2,2,3,3,2,2,4,4,3,3,0
7,2,1,0,0,0,2,4,3,3,3,2,4,5,1,7,3,5,2,7,5
8,1,1,0,4,2,3,3,4,2,3,3,4,4,4,1,5,3,2,2,5
9,2,0,2,1,1,2,3,3,3,2,2,2,2,3,0,2,1,1,1,1


count of repetitive sentences ↓


,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19
0,6,5,7,7,5,5,7,8,7,8,7,8,8,7,6,7,6,6,7,6
1,6,8,3,3,5,6,9,7,6,8,6,7,9,8,8,8,9,6,8,6
2,7,7,4,6,7,8,4,5,8,9,8,9,12,10,12,8,9,9,7,10
3,9,7,7,5,7,7,4,6,9,9,9,7,8,8,8,8,8,10,9,8
4,9,9,9,9,8,8,8,6,7,9,10,10,10,9,10,9,9,8,13,11
5,7,7,8,7,9,10,11,9,6,6,7,8,5,8,5,6,7,5,3,6
6,7,8,8,7,8,13,10,12,11,12,14,11,10,6,9,13,14,12,9,7
7,7,9,9,9,12,10,13,13,16,14,13,17,16,14,18,19,19,19,17,16
8,9,7,9,10,10,13,10,9,9,8,7,14,10,10,11,12,14,12,14,16
9,7,5,8,7,9,8,10,11,9,9,7,10,12,14,15,13,16,17,16,17


average perplexity of continuations ↓


,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19
0,2.54,2.84,2.74,2.74,2.89,2.80,2.79,2.71,2.75,2.77,2.87,2.87,2.88,2.89,2.79,2.85,2.89,2.90,2.86,2.89
1,2.57,2.55,2.66,2.82,2.85,2.92,2.99,3.05,3.04,2.87,2.84,3.11,3.01,3.09,3.21,3.16,3.16,3.12,2.98,3.00
2,2.58,2.61,2.78,2.86,2.76,2.83,2.85,2.70,2.82,2.85,2.94,2.98,3.13,3.05,3.13,3.37,3.32,3.30,3.70,3.05
3,2.52,2.73,2.67,2.72,3.30,2.88,2.79,2.88,2.76,2.73,2.73,2.70,2.80,2.81,2.79,2.87,3.14,3.17,3.14,3.57
4,2.57,2.51,3.31,2.81,3.43,4.86,3.54,3.35,3.08,3.55,3.37,3.46,4.31,3.56,3.31,2.96,3.03,4.43,3.81,4.09
5,2.61,2.64,2.71,2.82,2.87,2.88,2.72,2.74,2.74,2.75,2.72,2.73,2.89,2.91,3.13,3.25,2.97,2.95,3.06,3.01
6,2.57,2.57,2.79,2.69,2.71,2.64,2.49,2.54,2.43,2.50,2.51,2.74,2.77,2.94,2.82,2.76,2.76,2.87,2.94,2.74
7,2.48,2.64,2.63,2.58,2.49,2.57,4.14,3.25,5.39,6.16,6.13,11.17,13.40,15.77,13.88,15.61,16.42,17.17,14.21,25.76
8,2.53,2.50,2.58,2.59,2.60,2.68,2.89,2.91,2.85,2.92,2.94,2.85,3.02,3.10,3.36,3.14,3.04,3.18,3.38,3.45
9,2.55,2.70,2.55,2.48,2.56,2.77,2.95,2.88,2.89,2.75,2.88,2.83,2.76,2.62,2.70,2.75,2.70,2.83,3.48,3.30


### opt

In [13]:
for dir in dirs_opt:
    senti_stats(dir)

showing result for directory gemini_2pos_opt_senti+_fl_temp_0_no_space_hpt
count of positive continuation ↑


,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19
0,4,5,5,5,6,7,6,5,6,5,5,4,4,4,4,5,6,7,7,8
1,7,6,6,5,7,6,7,5,6,7,4,5,7,5,8,8,1,2,3,4
2,6,6,9,3,2,1,1,0,0,1,0,0,0,0,1,0,0,0,0,0
3,2,7,3,1,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
4,7,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
5,4,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
6,8,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
7,5,3,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
8,4,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
9,7,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0


count of repetitive sentences ↓


,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19
0,10,7,6,9,9,11,9,11,14,10,8,11,11,12,13,11,11,12,13,11
1,5,9,8,9,14,16,18,20,18,20,20,20,20,19,20,20,20,20,20,20
2,9,14,19,19,20,20,20,20,19,20,18,19,19,17,19,19,18,18,18,15
3,16,20,17,19,18,15,18,15,15,14,15,18,19,20,20,18,17,17,20,20
4,7,19,20,20,20,20,19,18,17,19,17,14,17,17,13,13,14,16,17,17
5,10,9,10,18,20,18,20,20,20,19,20,20,20,19,18,20,18,20,20,20
6,10,19,19,19,20,20,20,20,20,20,20,20,20,20,20,20,20,20,20,20
7,10,19,20,20,20,20,20,20,20,20,20,20,20,20,20,20,20,20,20,20
8,9,20,20,20,20,20,20,20,20,20,20,20,20,20,20,20,20,20,20,20
9,11,18,20,20,19,20,20,20,20,20,20,20,20,20,20,20,20,20,20,20


average perplexity of continuations ↓


...

,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19
0,4,3,5,5,5,4,6,6,6,6,5,4,4,4,4,2,2,3,3,3
1,4,4,5,5,6,5,7,5,4,1,1,3,3,4,3,4,2,2,2,1
2,4,5,4,5,5,3,3,0,1,1,2,2,1,2,0,0,1,0,0,0
3,4,3,3,5,3,3,1,3,2,0,1,0,0,0,0,0,0,0,0,0
4,4,2,4,2,2,1,1,0,0,0,1,0,0,0,0,0,0,0,0,0
5,4,3,1,2,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
6,4,3,2,1,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
7,3,3,2,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
8,5,3,3,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
9,4,4,3,3,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0


count of repetitive sentences ↓


,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19
0,10,8,10,10,12,9,10,12,12,12,12,13,11,11,11,10,10,10,10,12
1,10,10,14,14,14,14,16,15,13,13,12,13,12,11,12,8,10,12,12,12
2,10,10,11,7,15,14,16,17,16,18,16,17,18,17,18,17,20,20,20,20
3,9,7,10,12,12,17,15,18,19,14,13,8,8,4,4,3,3,3,3,3
4,10,12,13,15,17,18,20,20,19,20,20,19,19,19,18,19,17,19,17,16
5,12,7,10,16,14,18,14,15,14,15,15,12,14,12,13,11,13,13,10,10
6,9,11,14,18,18,20,19,20,18,14,16,17,17,17,18,17,19,19,18,19
7,11,9,12,17,20,17,19,20,20,19,19,20,19,20,20,19,19,19,19,18
8,11,13,15,18,19,19,20,18,19,19,19,19,17,16,17,15,15,15,12,14
9,12,11,12,19,18,20,20,19,20,20,20,20,19,20,20,19,19,20,20,20


average perplexity of continuations ↓


,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19
0,3.21,3.40,3.68,3.49,3.51,3.77,3.54,3.58,3.53,3.17,3.14,3.03,3.17,3.01,3.05,3.07,3.07,3.17,3.06,3.10
1,3.20,3.53,3.63,3.58,3.38,3.60,3.27,3.29,3.32,127.01,126.98,27.43,27.41,27.50,27.73,151.57,108.42,108.51,108.17,108.41
2,3.32,3.49,3.41,3.83,127.61,3.22,2.99,2.71,2.93,2.56,2.69,2.85,3.53,3.56,3.30,3.31,3.36,3.02,2.76,16.25
3,3.45,3.28,3.26,3.53,3.23,3.04,3.40,3.15,2.89,129.97,"9,806.65","82,611.40","3,978,861.52","3,361,772.97","6,194,701.33","151,697,431.25","1,541,399.67","43,858.55","202,289.89","208,520.67"
4,3.34,3.36,3.40,3.09,3.90,3.91,4.13,5.53,3.59,4.27,7.25,"32,761.81","43,398.04",348.01,"79,080,887.58","29,607,927.85","29,691,753.07","20,104.72","16,247,690.48","4,205,337.09"
5,3.36,3.35,3.00,3.67,"115,120.54","8,769.41","1,757,269.48","378,543.96","5,746,442.03","16,808,984.12","82,698,028.67","91,438,657.32","89,110,098.05","95,842,101.85","95,589,450.93","81,356,384.74","168,699,317.74","192,957,370.47","194,915,944.03","198,125,493.61"
6,3.36,3.13,2.96,2.47,2.78,2.49,5.05,"52,202.73","52,853.72","121,811.43","178,391.90","291,104.28","200,715.21","290,544.23","382,637.61","137,208.43","37,690.09","1,060.52","9,467.57","9,716.95"
7,3.19,3.35,3.41,3.17,11.59,5.43,5.31,66.77,255.14,"84,363.33","334,915.85","352,849.89","513,870.84","555,246.37","607,031.09","606,861.88","606,862.36","597,055.46","554,668.00","549,652.38"
8,3.33,3.20,3.12,3.73,18.36,64.50,"2,656,356,520.71","2,656,356,498.40","716,918,245.81","161,070,230.15","716,917,826.02","161,069,940.56","161,069,774.91","161,071,589.42","91,876,622,405.88","91,906,547,127.87","95,176,227,305.96","95,176,227,360.49","95,015,157,711.41","95,015,157,838.44"
9,3.23,3.29,3.68,5.36,4.03,2.72,2.93,2.82,2.76,4.60,5.07,8.73,13.32,12.81,13.72,14.10,15.45,10.86,16.14,13.77


### de

In [14]:
for dir in dirs_de:
    senti_stats(dir)

showing result for directory gemini_2pos_de_fl_senti+_temp_0_no_space
count of positive continuation ↑


,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19
0,7,8,8,7,9,9,9,11,9,11,7,7,7,6,7,6,6,8,6,5
1,7,9,10,3,1,1,2,3,1,1,1,2,2,4,3,3,3,2,2,3
2,5,4,7,4,8,9,6,7,8,9,6,7,4,4,5,7,5,7,6,4
3,6,2,4,8,6,9,9,7,8,11,12,9,10,9,11,8,8,10,10,9
4,6,6,5,10,11,9,8,12,8,10,11,11,8,9,13,14,14,15,14,11
5,3,6,5,5,7,8,9,9,9,10,9,12,13,9,9,8,11,12,12,14
6,6,7,6,6,8,5,5,5,5,7,8,7,9,11,11,10,9,9,8,8
7,5,4,5,5,7,10,7,8,8,9,9,6,6,6,8,8,8,8,5,3
8,4,3,4,5,6,6,5,4,5,8,9,7,7,8,9,8,7,10,11,9
9,2,3,5,5,5,7,7,8,7,6,6,6,5,6,6,6,7,7,7,8


count of repetitive sentences ↓


,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19
0,14,12,14,12,12,10,11,12,11,12,8,10,13,12,12,12,14,14,13,13
1,11,13,18,20,19,20,20,20,20,20,20,20,20,20,20,20,20,20,20,20
2,14,15,14,15,17,19,17,18,18,18,19,19,18,19,20,20,20,20,20,20
3,12,13,13,15,11,12,11,16,15,16,19,18,19,19,18,20,20,20,19,19
4,16,15,17,11,12,12,14,17,17,18,17,16,17,18,19,18,19,19,19,20
5,15,16,10,12,8,10,16,13,13,13,14,16,17,15,17,17,17,15,17,17
6,15,16,16,12,10,13,15,13,16,14,13,15,14,15,13,13,14,14,12,15
7,14,14,15,13,11,12,10,12,11,13,13,14,11,14,14,14,13,14,15,14
8,14,13,12,11,12,11,12,9,10,13,12,13,13,14,15,14,13,13,15,16
9,15,13,14,13,12,12,9,10,9,10,12,12,11,9,9,10,11,11,10,11


average perplexity of continuations ↓


...

,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19
0,0,1,0,0,0,0,0,1,1,1,2,2,1,1,1,2,2,2,2,2
1,0,1,0,1,1,2,1,1,1,1,0,1,0,0,0,0,0,0,1,1
2,0,0,1,1,1,2,1,0,0,0,1,0,0,0,0,0,0,0,0,0
3,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0
4,0,0,0,0,0,0,0,1,2,2,1,1,0,0,2,1,0,1,0,0
5,0,1,0,1,1,1,1,0,0,0,0,0,0,0,0,0,0,0,0,1
6,0,1,1,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0
7,0,0,0,2,1,2,2,1,0,0,0,1,0,0,0,0,4,3,3,2
8,0,0,0,0,1,2,2,2,3,2,2,2,2,3,2,2,3,3,1,1
9,0,0,0,0,0,0,0,0,0,1,1,1,2,1,0,0,1,1,3,2


count of repetitive sentences ↓


,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19
0,16,15,15,14,16,15,15,16,17,16,14,14,12,12,13,13,12,10,10,11
1,15,14,13,15,12,12,12,11,10,11,12,12,13,13,13,12,11,9,11,13
2,14,12,10,13,11,11,12,14,15,16,17,17,16,13,17,17,17,16,16,17
3,15,16,14,13,13,11,12,13,14,14,14,17,15,13,14,13,15,15,15,16
4,13,13,14,13,10,12,12,10,10,12,13,17,15,15,15,15,14,14,13,13
5,14,16,14,13,12,13,12,11,10,9,10,9,10,10,11,11,10,11,12,14
6,15,16,15,11,11,15,14,12,11,12,13,12,10,8,8,9,8,9,12,10
7,15,16,13,14,14,15,16,12,11,14,13,16,14,16,16,16,17,17,18,16
8,14,15,15,14,14,14,12,13,12,12,11,12,12,12,12,15,15,16,15,16
9,15,15,16,15,16,16,13,10,10,10,12,10,11,15,11,13,13,14,15,18


average perplexity of continuations ↓


,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19
0,2.91,2.90,3.03,2.93,2.79,2.79,2.78,2.78,2.80,2.74,2.84,2.81,2.82,2.90,2.95,2.95,2.93,2.96,3.02,3.01
1,2.87,2.88,2.99,2.86,2.86,2.80,2.78,2.86,2.77,2.84,2.79,2.84,2.82,2.73,2.71,2.80,2.76,2.80,2.78,2.78
2,2.93,2.94,2.97,2.93,3.08,2.94,2.89,2.87,2.92,3.01,2.78,2.72,2.75,2.90,2.92,2.81,2.74,2.65,2.72,2.77
3,2.91,2.92,2.82,2.94,2.83,2.87,2.73,2.71,2.66,2.59,2.57,2.70,2.69,2.78,2.77,2.75,2.66,2.76,2.77,2.75
4,2.89,3.03,3.24,4.04,6.60,3.49,3.48,7.23,5.09,6.01,4.03,7.12,82.54,12.49,6.39,4.41,9.70,8.20,11.65,17.08
5,2.87,3.07,2.98,2.82,2.84,2.81,2.83,2.86,2.85,2.88,2.90,2.90,2.91,2.89,3.00,3.04,3.15,3.15,2.99,2.95
6,2.87,2.98,2.81,2.79,2.82,2.79,2.83,2.89,2.98,2.99,2.95,2.87,2.88,2.96,2.91,3.08,3.02,2.97,2.95,3.09
7,2.89,2.86,2.87,2.77,2.78,2.94,3.01,2.94,2.93,2.93,2.96,2.88,2.91,2.87,2.95,3.04,3.37,3.57,3.75,4.49
8,2.91,2.91,2.83,2.91,3.00,3.09,2.99,3.07,3.25,3.26,3.36,3.32,3.39,3.23,3.23,3.12,3.18,3.21,3.29,2.96
9,2.90,2.90,2.79,2.85,2.77,2.89,2.93,2.88,2.99,3.05,3.06,3.16,3.27,3.23,3.15,3.02,3.07,3.15,3.13,3.03


### zh

In [15]:
for dir in dirs_zh:
    senti_stats(dir)

showing result for directory gemini_2pos_batch_zh_fl_senti+
count of positive continuation ↑


,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19
0,8,6,5,4,3,1,4,2,1,1,2,4,3,2,2,1,3,3,2,2
1,5,5,7,10,7,8,6,6,4,7,3,3,4,3,4,5,4,4,3,3
2,4,9,7,12,6,5,5,6,6,5,7,5,4,4,6,7,8,9,11,8
3,6,5,6,4,7,8,5,6,6,10,7,7,10,7,3,6,7,8,7,7
4,5,5,6,4,5,6,7,6,7,5,6,4,5,5,4,5,8,9,9,10
5,7,7,5,6,9,9,8,6,7,9,10,8,8,9,9,7,6,7,4,4
6,4,3,3,3,3,4,6,7,12,9,8,8,8,8,7,8,7,7,8,6
7,6,4,3,4,4,2,4,5,3,3,3,4,5,5,5,6,6,6,6,4
8,4,5,4,5,7,7,5,5,5,6,6,5,6,6,6,7,9,9,7,6
9,6,5,3,4,4,4,4,4,3,3,4,4,5,5,5,3,3,4,4,4


count of repetitive sentences ↓


,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19
0,16,15,18,18,19,18,18,18,19,19,20,18,18,18,19,19,20,20,20,20
1,15,10,15,13,12,12,13,14,12,13,10,14,13,15,16,18,14,13,11,6
2,18,16,17,13,16,18,16,17,16,16,16,14,13,15,13,10,11,10,14,14
3,18,14,15,17,14,15,14,15,15,15,16,14,14,15,14,15,17,15,14,15
4,17,17,16,14,16,15,13,13,13,17,17,17,14,17,16,16,16,16,17,17
5,16,18,15,14,17,14,16,18,17,18,16,15,17,16,16,18,20,16,16,19
6,17,16,16,15,13,12,17,16,16,16,17,15,17,17,16,15,15,15,16,17
7,16,16,19,15,15,16,14,15,16,17,16,15,16,16,16,16,17,17,17,17
8,17,14,16,17,17,16,15,17,18,17,18,18,17,18,19,18,17,18,18,19
9,14,15,15,14,14,17,17,15,17,18,19,19,18,18,17,18,17,18,18,19


average perplexity of continuations ↓


...

,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19
0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
1,0,0,1,1,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0
2,0,0,0,1,1,1,0,0,0,0,0,0,0,0,0,1,0,0,0,0
3,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
4,1,1,1,0,0,0,2,0,0,0,0,1,1,1,1,1,1,1,1,1
5,1,1,0,0,0,0,1,0,0,1,0,1,1,2,0,1,0,0,0,0
6,2,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
7,1,1,1,0,3,0,1,1,1,1,0,1,0,0,0,0,0,1,0,0
8,0,0,0,0,0,0,0,1,1,1,0,0,0,0,0,0,0,0,0,0
9,1,0,0,0,1,2,2,1,1,0,1,0,0,0,0,1,2,2,2,1


count of repetitive sentences ↓


,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19
0,15,16,15,17,17,15,15,17,15,17,17,14,14,15,14,13,15,15,17,18
1,17,16,17,15,17,15,17,16,17,16,17,15,16,16,14,16,16,14,16,16
2,18,17,17,18,18,17,16,17,16,15,16,16,18,18,14,16,16,13,16,18
3,16,16,15,16,13,17,17,16,13,16,15,16,16,17,15,16,18,17,19,16
4,16,16,14,17,15,19,15,15,15,14,15,17,15,17,17,18,17,18,17,17
5,16,17,18,16,19,18,18,17,18,20,19,19,19,18,18,18,17,18,18,18
6,16,16,19,18,18,17,17,18,16,17,16,18,16,17,15,16,16,14,16,17
7,16,18,17,19,19,16,14,15,16,15,18,17,18,18,19,18,18,19,19,20
8,17,17,17,16,19,19,18,19,19,17,16,16,17,17,18,17,17,18,18,16
9,17,18,17,16,16,16,15,12,12,16,16,16,16,15,16,16,15,14,13,14


average perplexity of continuations ↓


,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19
0,3.83,3.45,3.61,3.50,3.64,3.63,3.61,3.82,3.95,4.02,4.09,4.23,4.13,4.48,4.76,4.99,5.56,5.35,5.43,4.82
1,3.82,3.62,3.75,4.17,3.92,5.19,5.57,6.97,5.15,4.42,4.30,3.81,4.81,4.58,4.52,7.46,6.17,6.26,7.15,5.61
2,3.80,3.61,3.77,3.90,4.09,4.21,4.12,3.67,3.71,4.27,4.16,4.08,4.37,4.13,3.95,4.10,4.02,4.75,4.70,5.91
3,3.86,3.79,3.75,3.43,3.88,3.85,3.93,3.57,4.20,4.23,4.09,4.03,4.20,4.30,6.12,6.56,6.31,6.66,5.89,8.41
4,3.83,3.97,3.82,4.19,4.21,3.84,4.53,4.89,4.97,5.13,4.74,5.28,4.93,4.86,4.87,4.94,5.37,5.06,5.01,4.71
5,3.83,3.92,4.28,4.50,4.21,4.29,4.42,4.62,4.55,5.59,6.93,9.05,5.84,8.48,7.91,8.01,12.17,13.54,10.29,17.95
6,4.22,3.73,3.74,3.71,3.93,4.09,4.39,4.38,4.84,4.73,4.56,5.69,5.01,5.43,7.96,4.71,4.81,4.87,5.39,5.09
7,3.74,4.00,3.93,4.60,6.68,5.60,5.89,8.65,7.76,12.21,10.80,13.89,8.91,9.05,10.43,10.55,11.41,14.53,12.60,12.19
8,3.96,4.10,4.46,4.98,4.57,4.55,3.98,4.21,4.31,4.48,4.47,4.02,4.30,4.03,4.16,4.50,4.47,4.67,6.45,5.81
9,3.95,4.16,3.97,3.88,4.43,4.13,4.34,4.57,4.52,4.92,4.92,4.78,4.72,4.96,4.73,5.13,5.02,5.26,6.17,5.25


## comparing with baseline

In [16]:
# base_llama_sentimap[10].item()  # 0, 1, -1
def comparative_stats(dir, sentimap):
    """
    base_llama_sentimap or base_opt_sentimap 
    gemini_2pos_llama_senti+_fl_temp_0_no_space_hpt
    """
    print("showing result for directory", dir)    
    lst_files = os.listdir(f"{result_path}{dir}/")
    n_file = len(lst_files)
    grid_success = pd.DataFrame(0, index=range(n_file),columns=range(20))
    if "2pos" in dir:  # count the tags that are larger than the corresponding one in the base_map
        for file_name in lst_files:
            layer = int(file_name[file_name.rfind('_')+1:].split(".")[0])
            with open(f"{result_path}{dir}/{file_name}", "r") as f: 
                r_dict = json.load(f)
            for coeff in r_dict:
                list_dict = pd.DataFrame(r_dict[coeff])
                coeff = int(coeff)
                # pointwise compare with llama_sentimap
                successs = list_dict["continuation_label"] > sentimap
                grid_success.loc[layer, coeff-1] = successs.sum()
    if "2neg" in dir:  # count the tags that are smaller than the corresponding one in the base_map
        for file_name in lst_files:
            layer = int(file_name[file_name.rfind('_')+1:].split(".")[0])
            with open(f"{result_path}{dir}/{file_name}", "r") as f: 
                r_dict = json.load(f)
            for coeff in r_dict:
                list_dict = pd.DataFrame(r_dict[coeff])
                coeff = int(coeff)
                successs = list_dict["continuation_label"] < sentimap
                grid_success.loc[layer, coeff-1] = successs.sum()
    display(grid_success.style.background_gradient(cmap='Blues', axis=None))


### llama

In [17]:
for dir in dirs_llama:
    comparative_stats(dir, base_llama_sentimap)

showing result for directory gemini_2pos_llama_senti+_fl_temp_0_no_space_hpt


,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19
0,7,7,5,5,7,7,4,8,5,5,7,3,5,7,3,3,4,4,6,4
1,6,6,6,4,4,2,5,4,4,4,4,4,3,4,3,3,3,3,3,3
2,7,4,4,6,8,5,7,6,7,7,5,8,7,8,9,7,8,7,8,8
3,4,6,6,5,6,6,7,7,7,9,11,10,11,10,7,8,10,7,10,8
4,3,5,5,9,7,5,5,5,7,6,6,7,6,8,6,8,8,8,10,7
5,2,6,5,7,8,6,5,5,5,5,6,6,7,7,8,8,9,9,9,9
6,5,4,6,4,7,8,7,7,7,5,5,6,6,6,6,7,7,7,6,6
7,1,4,3,4,6,7,7,5,6,6,7,9,6,6,4,4,5,6,6,4
8,3,4,5,5,4,5,7,7,6,9,8,8,8,5,5,4,5,5,5,6
9,3,4,4,3,7,4,7,7,6,6,4,5,5,5,5,6,6,6,5,5


showing result for directory gemini_2neg_llama_senti+_fl_temp_0_no_space_hpt


,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19
0,1,4,3,6,5,5,5,5,5,5,5,5,5,5,5,5,5,3,2,2
1,6,4,4,2,2,5,5,6,4,4,4,4,3,4,6,4,3,3,4,5
2,2,5,5,4,5,4,3,7,6,5,4,5,3,4,4,6,3,5,4,4
3,3,6,6,8,6,6,6,6,7,6,5,5,6,7,9,6,5,9,8,8
4,1,2,2,4,5,5,6,5,5,5,5,5,6,5,5,4,5,5,6,7
5,2,3,2,2,3,3,4,4,5,4,5,5,7,7,7,7,7,7,7,6
6,3,3,4,3,4,4,5,5,4,5,3,3,3,4,5,4,4,4,3,3
7,2,3,3,3,6,4,3,5,5,7,7,7,6,6,5,5,4,4,4,3
8,2,2,4,3,4,3,4,5,5,5,5,4,5,5,5,5,4,5,5,5
9,2,2,1,4,7,6,5,5,4,6,5,4,4,5,5,6,6,6,7,6


showing result for directory gemini_sent_2pos_llama_senti+_fl_temp_0_hpt


,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19
0,2,5,5,4,5,4,4,3,2,3,3,4,5,4,4,4,3,3,3,4
1,1,7,5,5,4,3,2,3,5,5,6,6,5,3,4,3,5,3,3,4
2,2,5,7,6,7,4,5,9,6,6,8,5,3,4,1,4,3,3,3,3
3,2,4,6,8,5,3,5,7,6,6,6,7,7,8,8,9,8,6,6,6
4,2,4,6,6,6,7,5,5,8,6,5,8,9,7,6,8,8,8,8,9
5,5,6,5,5,3,4,5,4,7,9,8,9,5,6,5,3,5,3,4,6
6,4,5,4,6,5,5,5,8,8,8,6,6,4,5,7,10,8,7,6,8
7,3,3,7,6,8,8,7,7,8,6,7,7,5,6,7,6,5,5,6,6
8,5,5,4,6,3,6,7,6,5,5,7,4,4,5,6,5,6,6,7,6
9,4,3,6,6,8,4,6,6,6,9,8,7,8,8,8,6,7,7,8,6


showing result for directory gemini_sent_2neg_llama_senti+_fl_temp_0_hpt


,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19
0,1,3,5,7,7,5,4,5,4,4,4,4,4,4,4,5,4,4,4,4
1,2,4,3,8,5,5,7,5,6,5,5,6,6,6,6,6,6,6,7,7
2,2,5,4,5,7,4,4,4,4,6,6,6,6,7,7,6,7,5,5,4
3,0,2,2,2,2,2,4,5,5,4,3,4,5,5,6,6,6,6,6,5
4,0,1,4,4,5,4,4,4,4,5,7,8,7,6,7,5,5,7,7,5
5,1,4,4,4,5,4,4,5,4,4,6,6,6,5,7,6,5,7,6,4
6,1,2,3,3,3,6,5,5,6,5,5,6,6,6,6,8,8,7,7,4
7,2,4,4,4,4,6,7,7,6,6,4,8,9,5,9,7,10,7,11,9
8,1,2,4,8,6,8,6,7,4,6,7,8,8,7,6,9,6,6,6,6
9,1,1,4,5,6,7,5,5,6,7,7,7,7,8,5,7,6,4,3,5


### opt

In [18]:

for dir in dirs_opt:
    comparative_stats(dir, base_opt_sentimap)

showing result for directory gemini_2pos_opt_senti+_fl_temp_0_no_space_hpt


,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19
0,3,4,5,5,5,4,5,3,4,3,3,3,3,4,4,4,4,4,4,6
1,4,2,4,3,7,6,6,6,7,8,6,8,7,6,9,6,5,5,5,6
2,4,4,8,5,5,4,4,4,3,5,4,4,4,4,4,4,4,4,4,4
3,2,8,4,5,4,4,4,4,4,4,4,4,4,4,4,4,4,4,4,4
4,4,4,4,4,4,4,4,4,4,4,4,4,4,4,4,4,4,4,4,4
5,3,4,4,4,4,4,4,4,4,4,4,4,4,4,4,4,4,4,4,4
6,3,5,4,4,4,4,4,4,4,4,4,4,4,4,4,4,4,4,4,4
7,3,5,4,4,4,4,4,4,4,4,4,4,4,4,4,4,4,4,4,4
8,1,4,4,4,4,4,4,4,4,4,4,4,4,4,4,4,4,4,4,4
9,2,4,4,4,4,4,4,4,4,4,4,4,4,4,4,4,4,4,4,4


showing result for directory gemini_2neg_opt_senti+_fl_temp_0_no_space_hpt


,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19
0,4,4,7,5,4,4,5,2,2,2,2,4,3,5,4,3,2,5,4,3
1,3,3,3,3,5,4,6,8,7,5,6,7,7,7,7,7,7,7,7,7
2,4,3,7,6,7,8,5,7,7,7,7,7,7,7,7,7,7,7,7,7
3,3,7,7,7,7,7,7,7,7,7,7,7,7,7,7,7,7,7,7,7
4,4,5,7,7,7,7,7,7,7,7,7,7,7,7,7,7,7,7,7,7
5,5,6,7,7,7,7,7,7,7,7,6,7,5,7,7,7,7,7,7,7
6,3,5,4,3,4,6,7,7,7,7,7,7,7,7,7,7,7,7,7,7
7,5,7,7,6,7,7,7,7,7,7,7,7,7,7,7,6,7,7,7,7
8,4,5,7,7,6,7,7,7,7,7,7,7,7,7,7,7,7,7,7,7
9,6,6,6,7,7,7,7,7,7,7,7,7,7,7,7,7,7,7,6,6


showing result for directory gemini_sent_2pos_opt_senti+_fl_temp_0_hpt


,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19
0,0,5,3,4,4,4,3,3,3,2,3,3,2,2,2,2,2,2,2,2
1,0,2,3,5,4,3,3,4,5,4,5,3,5,6,2,3,5,7,6,7
2,0,2,3,5,5,4,5,8,5,6,6,6,5,5,4,5,5,4,3,3
3,0,2,4,6,4,3,4,4,4,4,4,4,4,4,4,5,4,4,3,3
4,1,1,4,5,5,4,4,3,1,1,1,2,3,4,4,4,4,4,4,4
5,3,2,4,4,4,3,4,4,4,4,4,4,4,4,4,4,4,4,4,4
6,2,2,5,3,5,4,4,4,4,4,4,4,4,4,4,4,4,4,4,4
7,2,2,5,6,4,4,4,4,4,4,4,4,4,4,4,4,4,4,4,4
8,1,2,6,5,4,4,5,4,4,4,4,4,4,4,4,4,4,4,4,4
9,0,3,7,4,4,4,5,5,5,4,4,4,4,4,4,4,4,4,4,4


showing result for directory gemini_sent_2neg_opt_senti+_fl_temp_0_hpt


,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19
0,0,4,9,7,8,7,6,6,6,6,5,6,5,4,4,4,4,3,3,3
1,0,6,8,9,7,6,8,5,4,3,4,3,3,4,4,6,6,5,2,4
2,1,7,7,6,6,6,7,3,3,5,5,6,3,7,6,6,6,6,7,7
3,3,5,4,8,5,6,4,7,7,7,5,7,7,7,7,7,7,7,7,7
4,4,3,7,4,4,4,5,4,5,6,5,7,7,6,7,7,7,7,7,7
5,3,2,3,3,4,4,5,7,7,7,7,7,7,7,7,7,7,7,7,7
6,4,2,3,5,5,5,7,7,7,7,7,7,7,7,7,7,7,7,7,7
7,4,3,3,3,4,7,7,7,7,7,7,7,7,7,7,7,7,7,7,7
8,4,4,5,4,6,7,7,7,7,7,7,7,7,7,7,7,7,7,7,7
9,4,5,4,5,5,7,7,7,7,7,7,7,7,7,7,7,7,7,7,7


### de

In [19]:
for dir in dirs_de:
    comparative_stats(dir, base_de_sentimap)

showing result for directory gemini_2pos_de_fl_senti+_temp_0_no_space


,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19
0,4,7,6,5,7,6,6,8,7,8,5,5,5,5,6,5,5,7,5,4
1,6,9,7,2,0,0,2,1,0,0,0,0,0,2,1,1,1,1,1,2
2,1,4,6,3,5,7,5,7,6,4,4,3,1,2,1,3,2,4,4,3
3,2,2,3,6,5,8,7,6,8,8,9,5,9,7,8,6,6,7,7,6
4,3,4,4,8,9,7,7,11,7,8,9,8,6,6,9,9,9,11,10,7
5,1,3,4,3,5,6,7,7,7,9,7,9,10,6,7,6,9,9,10,11
6,2,4,3,4,6,3,3,3,3,5,6,5,7,9,9,8,7,7,6,6
7,1,2,4,3,5,8,5,6,6,7,7,4,4,4,6,6,6,6,3,2
8,1,2,2,2,4,3,3,3,3,5,6,5,5,6,6,5,5,8,9,7
9,1,2,3,3,3,4,4,5,5,3,4,5,4,4,4,4,5,5,5,5


showing result for directory gemini_2neg_de_fl_senti+_temp_0_no_space


,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19
0,2,3,1,2,3,3,3,3,4,2,4,5,4,4,4,4,4,4,4,4
1,5,4,6,5,6,4,7,5,4,3,3,3,2,4,5,3,5,3,3,3
2,4,4,4,3,3,3,2,2,4,4,4,4,4,5,4,4,3,4,4,3
3,4,6,4,3,3,3,3,3,3,3,3,4,3,4,5,3,3,4,5,7
4,1,4,3,3,4,4,4,4,4,4,3,3,3,3,2,2,2,3,3,3
5,2,4,3,3,3,4,4,4,4,4,3,5,5,5,4,5,5,5,5,5
6,3,3,3,3,2,2,3,3,5,5,5,4,4,5,6,6,6,6,6,6
7,1,2,3,3,3,3,3,4,4,5,5,5,5,5,5,5,4,4,4,5
8,2,1,1,1,2,3,4,3,3,4,4,5,4,4,3,3,4,3,3,3
9,0,1,2,3,4,4,6,5,3,6,5,5,6,5,6,6,7,6,7,7


showing result for directory gemini_sent_2pos_de_fl_senti+_temp_0


,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19
0,2,3,3,4,2,2,2,3,3,3,3,3,3,3,2,2,2,3,3,3
1,4,2,3,3,3,3,1,2,4,4,3,4,3,5,6,5,7,8,6,7
2,2,4,1,1,5,4,3,5,4,2,2,4,1,2,1,0,1,1,0,0
3,0,2,2,2,0,1,2,3,2,2,2,3,3,4,3,4,3,4,4,4
4,3,1,2,2,3,4,6,6,7,3,2,3,4,5,7,7,6,7,7,8
5,5,0,1,5,5,5,4,5,6,3,3,3,4,3,5,4,3,1,1,2
6,4,5,3,3,2,4,7,9,9,7,7,8,7,8,7,8,8,6,5,8
7,2,2,5,7,5,4,6,5,8,7,6,7,5,7,5,8,6,8,6,7
8,1,3,3,1,1,2,2,2,4,5,6,4,6,7,8,7,7,6,5,4
9,1,3,5,2,4,5,6,6,3,3,3,2,3,3,5,6,5,6,4,3


showing result for directory gemini_sent_2neg_de_fl_senti+_temp_0


,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19
0,3,4,4,4,3,3,4,5,4,5,6,6,4,4,4,6,6,6,6,5
1,2,4,3,3,4,5,5,5,4,4,3,3,2,2,2,2,2,2,4,4
2,2,2,4,5,3,5,3,4,4,4,4,3,3,4,4,5,5,5,4,4
3,1,1,2,3,3,2,2,3,2,2,2,2,2,2,3,3,3,3,3,3
4,0,3,3,4,3,4,3,3,4,4,4,3,2,3,5,4,4,5,3,3
5,1,6,4,5,5,5,4,3,2,2,3,3,2,3,3,3,2,2,2,3
6,2,3,5,3,3,3,3,3,3,2,2,2,2,2,3,2,1,2,3,4
7,3,3,3,5,4,5,5,4,3,4,4,5,3,4,4,4,6,6,6,5
8,2,4,4,3,4,5,4,4,5,5,5,6,6,6,5,5,6,6,4,5
9,2,2,3,3,2,4,4,3,4,5,4,5,3,4,3,2,3,4,4,5


### zh

In [20]:
for dir in dirs_zh:
    comparative_stats(dir, base_zh_sentimap)

showing result for directory gemini_2pos_batch_zh_fl_senti+


,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19
0,5,4,4,5,3,1,2,1,1,2,3,5,4,3,3,2,3,3,3,3
1,3,3,5,7,5,5,5,5,3,4,3,3,3,3,2,4,4,3,3,4
2,1,7,4,8,3,2,2,3,3,3,5,4,3,1,4,4,5,5,8,6
3,3,4,4,3,6,6,4,5,4,7,6,7,7,7,3,5,6,7,6,6
4,4,3,5,3,4,4,4,3,4,4,5,4,6,4,3,4,6,7,7,7
5,5,4,4,4,7,7,6,5,5,7,8,7,6,6,7,6,6,6,4,4
6,2,1,2,2,2,3,4,5,8,6,6,5,6,6,6,6,4,4,5,4
7,4,2,1,2,3,1,2,4,2,2,2,2,3,3,3,4,4,4,4,4
8,2,3,2,2,4,4,2,2,4,4,5,4,4,4,4,4,5,5,4,4
9,3,3,1,2,2,2,2,3,2,2,2,2,3,3,3,2,2,3,3,3


showing result for directory gemini_2neg_batch_zh_fl_senti+


,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19
0,1,3,2,2,1,3,2,2,2,2,2,1,2,3,2,3,3,3,3,3
1,2,2,2,2,2,2,2,3,3,3,4,5,3,2,3,4,5,5,4,5
2,1,1,2,3,3,1,0,1,2,1,3,3,3,4,4,4,5,4,4,3
3,1,2,1,3,2,3,1,1,4,2,2,1,1,1,1,1,1,1,1,1
4,2,3,1,0,0,0,0,0,0,2,2,1,1,1,2,2,1,2,2,1
5,2,2,2,2,1,1,1,1,1,1,1,2,2,2,2,3,3,3,3,3
6,1,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2
7,1,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2
8,1,2,1,2,2,3,3,3,3,2,2,2,2,2,1,1,1,1,1,1
9,2,2,2,1,1,1,1,1,1,1,0,1,1,1,1,1,1,1,1,1


showing result for directory gemini_sent_2pos_batch_zh_fl_senti+


,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19
0,2,2,6,6,5,5,5,4,3,3,3,5,4,3,3,5,5,4,3,3
1,3,2,5,7,5,6,6,7,7,7,7,7,7,7,5,5,5,8,8,6
2,2,4,4,7,6,5,4,5,5,5,6,4,5,6,5,5,5,4,5,4
3,2,2,2,4,3,5,5,6,7,7,8,6,6,6,5,5,5,4,4,4
4,1,3,3,4,5,4,4,5,3,4,6,4,3,4,7,6,5,5,5,5
5,1,3,1,4,2,3,6,4,4,3,5,6,5,6,5,6,5,7,5,4
6,1,2,5,4,4,8,8,8,6,6,4,6,6,7,4,4,5,5,8,5
7,2,2,3,4,6,7,6,8,8,6,6,4,6,6,4,5,3,6,7,5
8,2,6,3,2,5,3,3,2,4,1,3,0,0,1,3,3,4,2,2,3
9,2,2,4,3,6,4,3,4,5,5,5,5,5,5,5,6,6,6,6,7


showing result for directory gemini_sent_2neg_batch_zh_fl_senti+


,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19
0,1,0,0,1,0,0,1,1,2,2,1,2,2,4,4,4,2,2,3,3
1,1,0,1,1,0,0,0,2,3,3,4,4,4,4,3,4,4,3,4,5
2,0,0,0,1,1,1,0,0,2,3,4,3,3,3,3,3,2,2,2,2
3,0,1,2,1,1,1,2,2,2,1,2,2,2,2,2,2,2,2,3,4
4,1,1,1,0,0,0,2,1,1,1,2,2,2,2,2,3,2,3,4,4
5,1,1,1,2,2,2,2,1,1,2,3,2,3,4,3,4,3,3,3,4
6,1,1,1,3,2,0,0,1,0,1,2,2,3,5,5,5,2,3,3,3
7,2,1,1,0,4,2,3,4,3,3,2,4,3,4,4,4,4,5,4,4
8,1,1,1,0,0,2,1,3,3,4,3,4,4,4,4,4,4,3,3,3
9,2,1,1,2,3,4,5,4,4,4,5,4,4,2,3,4,6,7,7,6


# talking about the bridge

In [23]:
def bridge_stats(dir):
    """
    counting the number of steered sentences that talk about the golden gate bridge 
    """
    print("showing result for directory", dir)
    lst_files = os.listdir(f"{result_path}{dir}/")
    n_file = len(lst_files)
    grid_bridge = pd.DataFrame(0, index=range(n_file),columns=range(20))
    grid_rep = pd.DataFrame(0, index=range(n_file),columns=range(20))
    grid_fl = pd.DataFrame(index=range(n_file),columns=range(20))
    for file_name in lst_files:
        layer = int(file_name[file_name.rfind('_')+1:].split(".")[0])
        with open(f"{result_path}{dir}/{file_name}", "r") as f: 
            r_dict = json.load(f)
        for coeff in r_dict:
            list_dict = pd.DataFrame(r_dict[coeff])
            coeff = int(coeff)
            if 1 in list_dict["bridge"].value_counts():
                grid_bridge.loc[layer, coeff-1] = list_dict["bridge"].value_counts()[1]
            else:
                grid_bridge.loc[layer, coeff-1] = 0
            grid_rep.loc[layer, coeff-1] = list_dict["repetition"].sum().item()
            grid_fl.loc[layer, coeff-1] = list_dict["fluency"].mean().item()
    # https://stackoverflow.com/questions/12286607/making-heatmap-from-pandas-dataframe
    # https://stackoverflow.com/questions/61363712/how-to-print-a-pandas-io-formats-style-styler-object
    
    print("count of sentences talking about the bridge ↑")
    display(grid_bridge.style.background_gradient(cmap='Blues', axis=None))
    print("count of repetitive sentences ↓")
    display(grid_rep.style.background_gradient(cmap='Reds', axis=None))
    print("average perplexity of continuations ↓")
    gmap_clipped = grid_fl.clip(upper=grid_fl.quantile(0.95), axis=1)
    display(
        grid_fl.style.background_gradient(cmap='Reds', gmap=gmap_clipped, axis=None).format("{:,.2f}")
    )

In [24]:
for dir in dirs_bridge:
    bridge_stats(dir)

showing result for directory gemini_bridge_llama_bridge+_fl_hpt
count of sentences talking about the bridge ↑


,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19
0,2,10,11,9,10,10,8,6,5,2,2,0,0,0,0,0,0,0,0,1
1,0,13,14,12,15,15,15,11,5,5,4,6,9,10,10,13,14,11,12,11
2,0,12,11,12,12,14,8,7,6,6,8,7,9,15,17,15,16,17,19,20
3,0,3,5,7,13,11,14,14,17,17,18,18,16,15,11,14,12,12,16,13
4,0,5,6,2,0,0,5,7,14,18,17,19,15,15,14,13,16,16,18,17
5,0,2,3,3,4,4,10,14,18,16,18,16,17,19,18,18,17,17,18,19
6,0,4,4,1,2,4,10,11,7,11,15,14,16,19,19,19,18,17,18,18
7,0,3,1,0,0,2,9,10,13,15,15,14,14,16,18,17,18,20,20,19
8,0,4,1,0,1,7,8,9,10,10,11,8,10,9,11,10,9,11,12,12
9,0,2,0,0,1,1,8,9,10,9,11,10,15,14,14,15,14,14,16,16


count of repetitive sentences ↓


,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19
0,5,11,14,18,18,19,20,20,20,20,20,19,20,20,20,20,20,20,20,20
1,7,4,14,16,19,19,20,19,20,18,20,20,20,20,20,20,20,20,20,20
2,8,6,8,13,12,11,10,11,10,11,11,14,12,18,19,20,20,20,20,20
3,10,7,10,8,11,8,6,5,6,10,3,7,10,16,20,19,20,20,20,19
4,7,7,10,10,10,13,8,12,18,20,19,20,19,19,20,20,20,20,20,20
5,8,10,9,7,15,13,13,15,19,20,20,20,19,19,19,19,20,20,20,20
6,5,11,8,8,4,9,14,14,16,17,16,17,19,19,18,18,18,19,19,19
7,9,10,10,9,13,14,16,16,14,17,18,19,19,18,19,20,20,20,20,19
8,5,7,10,4,7,11,12,15,18,17,16,18,18,19,20,20,20,19,19,19
9,10,6,8,11,8,9,9,11,14,17,18,18,18,18,19,19,19,18,17,18


average perplexity of continuations ↓


...

,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19
0,0,1,0,1,0,0,0,0,0,0,0,0,0,0,1,1,2,1,1,1
1,0,0,0,0,1,1,1,6,6,5,9,10,11,18,18,19,20,20,20,18
2,0,0,0,0,0,0,0,2,3,4,6,7,8,11,11,11,9,9,9,10
3,0,0,0,0,0,0,0,1,2,1,8,7,11,9,11,13,14,14,14,12
4,0,0,0,0,0,0,1,6,7,9,10,13,12,11,13,14,14,15,15,15
5,0,0,0,0,0,1,2,3,5,6,8,9,9,9,9,9,11,11,11,11
6,0,0,0,2,1,1,1,1,1,1,1,1,1,2,2,2,2,2,1,0
7,0,0,0,1,3,2,2,2,1,2,2,3,3,3,3,3,4,4,5,5
8,0,1,0,0,0,0,0,1,1,1,3,4,3,3,4,4,5,6,7,7
9,0,1,0,0,0,0,1,0,0,2,1,1,1,2,4,5,4,4,6,4


count of repetitive sentences ↓


,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19
0,16,13,16,18,20,19,20,20,19,19,19,20,19,18,18,18,17,17,16,16
1,17,17,16,16,16,19,18,12,13,18,13,9,5,2,2,2,1,0,0,1
2,13,15,16,16,14,17,19,18,19,19,20,20,19,18,19,19,19,19,19,19
3,17,14,14,12,15,16,18,17,20,20,20,20,20,20,20,20,20,20,20,20
4,16,11,17,15,13,13,17,14,16,16,16,20,18,17,18,19,20,20,20,20
5,19,13,12,13,13,17,14,15,13,13,17,16,16,15,15,18,19,20,17,17
6,18,16,15,14,14,14,15,14,14,14,14,15,15,18,19,18,16,16,17,18
7,18,15,13,13,13,14,14,15,15,14,15,14,14,13,13,15,15,12,13,12
8,16,16,14,14,17,14,14,15,14,14,14,13,14,14,13,16,15,16,14,15
9,17,17,17,17,14,18,16,17,16,17,17,18,18,17,15,15,14,14,13,13


average perplexity of continuations ↓


,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19
0,4.00,5.45,4.45,6.47,16.03,6.44,6.53,6.03,8.40,13.93,6.58,5.96,8.46,5.16,104.65,104.50,5.56,4.99,5.97,5.38
1,4.62,6.28,"4,324.62","4,324.39","4,325.54",28.17,135.93,"1,747.59","2,047.94","4,483,975.41","4,062,228,420.04","4,076,104,826.77","4,166,328,421.82","4,435,005,039.05","4,310,775,000.10","4,449,537,517.71","4,449,545,746.55","4,452,441,170.46","4,453,442,921.56","4,450,114,423.43"
2,4.59,5.46,7.15,5.04,5.39,11.81,6.77,9.02,204.01,6.80,10.27,10.20,8.76,7.94,6.76,8.24,11.88,7.36,7.22,12.95
3,4.35,5.39,5.89,5.53,5.75,5.75,4.70,5.02,4.80,5.43,4.77,6.29,5.30,5.13,4.92,7.12,10.35,9.69,10.40,8.14
4,4.12,5.36,5.20,6.29,7.67,7.58,6.93,8.07,8.24,8.59,9.14,8.21,8.42,11.94,11.37,10.94,12.80,15.08,14.33,17.17
5,4.47,5.40,5.79,5.50,7.64,6.43,7.78,10.76,8.80,8.94,9.20,10.00,11.66,8.72,8.57,8.41,7.26,7.94,8.27,7.48
6,4.57,4.69,5.09,6.68,6.92,6.12,5.61,5.35,5.27,5.30,5.32,5.24,5.33,5.51,5.73,6.33,6.49,6.48,6.54,6.72
7,4.69,4.87,5.74,6.37,6.27,6.18,5.38,5.89,5.75,6.83,6.06,6.15,6.27,6.10,6.58,6.27,6.98,6.83,7.56,7.69
8,4.60,4.93,4.53,4.67,4.70,4.73,4.90,4.92,5.72,6.49,6.40,7.08,6.47,6.03,6.76,7.81,7.94,8.74,8.07,9.02
9,4.82,4.75,5.05,4.90,4.56,4.78,4.66,4.57,5.21,5.28,5.55,6.23,6.31,6.07,6.00,6.84,6.93,7.68,7.20,7.10


# temperature=1
all the results above are for when temperature = 0
below are additional analysis for when temperature = 1, which includes
- steering with a pair of phrases with one white space prepended to each phrase
- steering with a pair of sentences

In [27]:
baseline_llama_temp1 = "gemini_base_llama_fl_senti.json"
baseline_opt_temp1 = "gemini_base_opt_fl_senti.json"

dirs_llama_temp1 = [
    "gemini_2pos_llama_senti_hpt", 
    "gemini_2neg_llama_senti_hpt", 
    "gemini_sent_2pos_llama_senti_hpt", 
    "gemini_sent_2neg_llama_senti_hpt"
    ]

dirs_opt_temp1 = [
    "gemini_2pos_opt_senti_hpt", 
    "gemini_2neg_opt_senti_hpt", 
    "gemini_sent_2pos_opt_senti_hpt",
    "gemini_sent_2neg_opt_senti_hpt"
    ]

In [28]:
def base_temp1(base_file):
    df = pd.read_json(baseline_path + base_file)
    print("counts of", df["continuation_label"].value_counts())
    print("average perplexity of continuations:", df["fluency"].mean().item())
    return df["continuation_label"]
base_llama_sentimap_temp1 = base_temp1(baseline_llama_temp1)
base_opt_sentimap_temp1 = base_temp1(baseline_opt_temp1)
print(base_llama_sentimap_temp1, base_opt_sentimap_temp1)

counts of continuation_label
 1    11
 0     8
-1     1
Name: count, dtype: int64
average perplexity of continuations: 22.038273668289186
counts of continuation_label
 0    12
 1     6
-1     2
Name: count, dtype: int64
average perplexity of continuations: 27.720147919654845
0     1
1    -1
2     0
3     0
4     1
5     1
6     0
7     1
8     1
9     0
10    1
11    1
12    1
13    0
14    0
15    1
16    1
17    1
18    0
19    0
Name: continuation_label, dtype: int64 0     1
1     1
2     1
3     0
4    -1
5     0
6     0
7     0
8     1
9     0
10    0
11    1
12   -1
13    0
14    0
15    1
16    0
17    0
18    0
19    0
Name: continuation_label, dtype: int64


## sentiment counting 1s or -1s


In [29]:
def senti_temp1_count(dir):
    print("showing result for directory", dir)
    lst_files = os.listdir(f"{result_path}{dir}/")
    n_file = len(lst_files)
    grid_one = pd.DataFrame(0, index=range(n_file),columns=range(20))
    grid_zero = pd.DataFrame(0, index=range(n_file),columns=range(20))
    grid_neg = pd.DataFrame(0, index=range(n_file),columns=range(20))
    for file_name in lst_files:
        layer = int(file_name[file_name.rfind('_')+1:].split(".")[0])
        with open(f"{result_path}{dir}/{file_name}", "r") as f: 
            r_dict = json.load(f)
        for coeff in r_dict:
            list_dict = pd.DataFrame(r_dict[coeff])
            coeff = int(coeff)
            if 1 in list_dict["continuation_label"].value_counts():
                grid_one.loc[layer, coeff-1] = list_dict["continuation_label"].value_counts()[1]
            if 0 in list_dict["continuation_label"].value_counts():
                grid_zero.loc[layer, coeff-1] = list_dict["continuation_label"].value_counts()[0]
            if -1 in list_dict["continuation_label"].value_counts():
                grid_neg.loc[layer, coeff-1] = list_dict["continuation_label"].value_counts()[-1]
    if "2pos" in dir:
        print("count of positive continuation ↑")
        display(grid_one.style.background_gradient(cmap='Blues', axis=None))
    elif "2neg" in dir:
        print("count of negative continuation ↑")
        display(grid_neg.style.background_gradient(cmap='Blues', axis=None))

### llama

In [30]:
for dir in dirs_llama_temp1:
    senti_temp1_count(dir)

showing result for directory gemini_2pos_llama_senti_hpt
count of positive continuation ↑


,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19
0,7,7,3,3,3,4,4,4,4,5,6,8,7,7,8,6,4,4,6,7
1,7,8,8,5,5,5,4,4,4,4,4,5,6,5,5,5,6,6,6,7
2,5,9,6,6,5,6,6,7,7,7,7,6,6,5,4,4,4,6,6,5
3,6,5,6,8,8,9,9,7,6,6,5,5,4,4,5,5,5,5,6,6
4,6,6,7,7,8,10,9,7,6,7,7,7,7,7,7,5,6,5,4,5
5,5,7,7,6,6,7,7,5,8,8,8,8,9,10,9,9,8,8,7,7
6,6,7,10,9,10,7,6,7,7,7,6,6,5,6,6,7,7,8,9,10
7,5,8,9,8,4,7,5,8,8,8,10,8,7,8,8,6,4,4,4,3
8,5,5,7,7,7,8,7,8,8,11,9,8,9,4,6,6,7,7,7,5
9,5,7,7,8,10,9,7,7,8,8,8,7,6,7,7,7,6,8,8,8


showing result for directory gemini_2neg_llama_senti_hpt
count of negative continuation ↑


,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19
0,4,0,1,0,1,1,1,1,1,1,1,0,0,1,1,0,0,0,0,0
1,3,3,4,5,4,4,4,3,4,3,3,6,2,3,5,3,3,2,1,0
2,4,5,6,5,3,4,5,1,1,4,1,2,1,2,0,3,2,3,3,2
3,0,3,3,3,2,2,3,5,4,2,2,2,2,4,5,6,2,4,5,6
4,2,2,2,4,4,4,4,3,4,3,2,4,2,2,3,1,1,2,1,1
5,2,1,2,3,3,2,3,2,3,3,3,2,3,4,3,3,2,3,2,3
6,2,1,1,3,2,2,2,2,2,2,2,1,4,4,3,2,2,2,3,3
7,1,2,1,2,4,3,3,3,3,4,3,1,1,1,2,3,4,2,3,2
8,1,3,2,2,3,4,4,4,2,1,1,1,1,3,2,1,1,2,2,2
9,2,2,2,2,2,3,2,1,3,1,3,2,3,3,2,2,4,4,5,4


showing result for directory gemini_sent_2pos_llama_senti_hpt
count of positive continuation ↑


,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19
0,5,7,6,6,6,8,8,8,7,6,6,6,7,7,7,7,8,7,7,8
1,7,6,4,7,8,6,5,6,10,6,6,3,3,5,7,7,10,10,10,9
2,7,5,6,6,6,6,5,2,5,2,1,2,2,1,0,2,2,0,0,3
3,6,3,6,5,7,8,8,9,8,7,9,8,11,11,12,11,11,9,11,8
4,9,5,7,10,5,9,10,10,6,8,9,9,9,6,5,5,8,3,5,5
5,4,6,8,9,10,9,11,8,8,5,4,5,2,1,4,5,4,0,4,2
6,4,8,8,7,10,7,10,12,11,10,7,11,13,13,11,11,13,10,6,8
7,4,8,8,7,9,10,9,7,7,11,11,8,8,6,6,12,10,11,8,6
8,5,7,8,8,8,10,11,9,8,8,6,11,10,7,6,10,13,8,10,12
9,4,5,6,8,7,8,9,7,6,4,5,9,10,11,11,10,9,8,12,12


showing result for directory gemini_sent_2neg_llama_senti_hpt
count of negative continuation ↑


,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19
0,4,1,3,3,2,1,1,1,1,0,0,0,0,0,0,0,0,0,0,1
1,4,2,4,3,3,2,3,2,1,0,1,0,0,0,1,2,2,3,4,2
2,1,1,2,3,4,3,2,3,1,0,2,2,2,2,1,1,2,1,1,0
3,4,2,2,5,6,5,3,2,2,2,2,1,0,1,2,1,1,1,1,1
4,2,5,3,4,3,3,3,5,4,4,3,3,2,3,1,0,3,5,5,5
5,3,5,4,4,4,4,3,3,2,1,1,2,1,1,2,1,2,2,3,3
6,2,5,5,5,4,4,4,3,3,2,2,5,4,3,3,4,4,3,3,5
7,2,6,5,5,3,3,2,3,4,1,2,2,2,2,3,4,6,3,5,3
8,3,4,3,3,6,4,6,6,4,4,3,2,4,5,7,3,4,4,4,5
9,5,5,5,2,2,3,3,3,4,3,6,4,4,3,3,7,4,4,5,6


### opt

In [31]:
for dir in dirs_opt_temp1:
    senti_temp1_count(dir)

showing result for directory gemini_2pos_opt_senti_hpt
count of positive continuation ↑


,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19
0,6,1,4,3,4,3,1,2,1,4,6,7,7,8,7,6,6,6,6,5
1,6,7,7,5,4,5,5,4,1,4,4,6,5,7,3,9,1,2,2,3
2,4,4,10,5,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
3,4,3,3,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0
4,4,5,3,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
5,3,4,1,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
6,4,6,3,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
7,5,5,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
8,6,4,4,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
9,5,4,4,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0


showing result for directory gemini_2neg_opt_senti_hpt
count of negative continuation ↑


,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19
0,2,3,4,5,4,3,3,3,4,2,2,2,3,2,2,4,3,2,2,2
1,3,3,3,4,3,3,4,4,3,3,0,1,1,1,1,2,1,3,1,3
2,3,5,2,5,3,3,0,1,0,0,0,0,0,1,0,0,0,0,0,0
3,3,2,4,2,2,2,3,3,5,2,1,0,0,0,1,1,1,0,0,0
4,3,1,4,2,2,3,2,1,1,0,0,1,0,0,0,0,0,0,0,0
5,3,3,2,1,1,2,3,4,2,1,0,1,0,0,1,0,0,0,0,0
6,3,4,3,3,0,1,1,1,1,3,0,1,0,0,0,0,0,0,0,0
7,3,4,3,3,4,1,4,4,2,1,1,2,1,1,0,0,0,0,0,0
8,1,2,2,3,0,4,2,2,3,0,1,1,0,0,1,0,0,0,0,1
9,2,2,2,4,1,1,0,2,1,1,1,1,1,0,0,0,0,1,1,0


showing result for directory gemini_sent_2pos_opt_senti_hpt
count of positive continuation ↑


,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19
0,8,6,5,6,7,9,7,8,7,6,6,6,6,7,7,7,5,5,5,4
1,7,7,5,10,6,9,9,9,10,9,9,6,7,5,7,8,6,6,9,7
2,7,6,5,5,4,3,2,5,0,3,4,3,1,1,1,1,0,0,0,0
3,4,4,3,1,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0
4,6,6,6,5,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0
5,4,7,5,6,3,3,2,0,0,1,0,0,0,0,0,0,0,0,0,0
6,3,6,5,5,1,1,0,0,0,0,0,0,0,0,0,0,0,1,1,0
7,4,5,6,2,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
8,3,5,4,1,2,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
9,4,6,3,2,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0


showing result for directory gemini_sent_2neg_opt_senti_hpt
count of negative continuation ↑


,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19
0,1,4,4,4,5,6,6,4,3,3,2,1,2,2,2,2,2,2,2,2
1,3,2,3,5,3,3,2,2,1,2,3,1,2,2,3,3,3,2,1,3
2,3,3,4,5,7,3,3,5,2,2,3,1,4,4,3,2,0,3,0,2
3,3,5,6,7,6,2,2,2,0,2,1,1,1,1,2,2,2,1,1,2
4,3,2,5,0,3,1,1,0,0,0,0,0,0,0,0,0,1,1,1,2
5,2,2,4,3,1,2,1,1,2,1,1,4,2,1,5,3,4,2,3,3
6,2,2,4,2,0,2,1,3,2,3,0,1,1,0,0,1,1,1,0,1
7,2,4,0,1,1,2,2,3,2,1,1,2,4,4,1,0,1,0,0,1
8,2,4,2,2,2,0,2,0,0,1,0,0,0,0,0,1,0,0,0,0
9,2,4,5,1,0,2,2,2,3,0,3,2,2,3,0,2,1,2,2,3


## sentiment comparing with the baseline
### llama

In [32]:
for dir in dirs_llama_temp1:
    comparative_stats(dir, base_llama_sentimap_temp1)

showing result for directory gemini_2pos_llama_senti_hpt


,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19
0,2,2,2,2,2,1,1,1,1,1,2,3,3,3,3,2,2,2,2,3
1,2,2,3,1,2,2,2,2,2,2,2,3,3,2,2,2,2,2,2,2
2,3,4,1,2,1,1,2,3,3,3,3,2,2,1,1,1,1,2,2,2
3,2,2,2,2,2,2,2,2,2,2,2,2,1,1,1,1,1,1,2,1
4,2,2,2,2,2,3,2,1,1,2,2,3,3,2,2,2,2,1,1,1
5,2,2,3,1,1,1,1,2,3,3,3,3,3,3,3,3,3,3,2,2
6,2,2,4,4,4,3,3,3,3,4,3,2,2,2,2,2,2,2,3,3
7,2,4,4,3,2,4,3,4,2,3,4,2,2,3,3,2,1,1,1,1
8,2,2,2,3,3,3,3,2,3,4,4,3,4,3,3,3,3,4,4,4
9,2,2,2,3,3,3,2,2,2,2,2,2,2,2,2,2,2,2,2,2


showing result for directory gemini_2neg_llama_senti_hpt


,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19
0,6,6,9,6,9,10,9,9,9,10,8,7,7,7,6,7,8,9,9,10
1,7,5,6,8,8,9,10,9,11,11,8,10,9,11,8,8,7,7,9,9
2,8,10,9,9,10,11,10,8,6,12,6,9,9,8,8,11,11,12,11,12
3,7,7,9,8,8,6,8,9,10,9,11,8,9,11,11,11,7,7,9,10
4,6,6,6,6,6,6,8,7,9,8,10,12,11,10,10,9,10,9,8,9
5,6,5,5,6,7,7,8,7,7,7,9,8,10,9,11,11,8,8,8,8
6,4,5,6,7,7,8,8,8,8,8,7,6,8,10,9,9,11,10,8,8
7,5,6,6,5,8,7,7,8,8,9,9,6,8,7,5,6,8,8,8,9
8,4,6,7,7,7,8,8,8,5,7,10,11,9,8,9,7,5,5,5,5
9,4,5,7,7,7,8,7,6,6,7,8,8,8,8,7,8,9,9,9,8


showing result for directory gemini_sent_2pos_llama_senti_hpt


,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19
0,2,1,2,3,3,3,2,2,2,2,2,2,2,2,3,3,3,3,3,3
1,2,2,1,3,3,3,2,2,3,2,3,1,2,3,3,3,3,3,3,3
2,2,2,2,2,2,3,2,1,3,2,1,2,2,2,1,2,1,1,1,2
3,3,1,3,3,4,4,3,4,3,3,4,2,4,4,5,4,3,3,4,4
4,4,2,2,4,3,6,5,3,3,3,4,4,4,3,2,3,3,1,2,2
5,2,2,3,1,3,3,4,4,5,3,3,4,2,2,2,2,2,1,2,2
6,2,2,3,2,3,2,5,6,4,6,5,5,5,5,3,6,6,5,3,5
7,2,4,2,2,4,4,3,2,3,6,5,4,4,3,3,8,5,5,3,3
8,2,2,3,4,3,3,5,3,4,4,3,5,4,3,3,4,6,4,5,6
9,2,2,3,2,1,2,2,2,2,2,2,6,6,4,4,4,2,3,4,5


showing result for directory gemini_sent_2neg_llama_senti_hpt


,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19
0,8,4,8,10,9,7,7,8,5,5,4,5,7,7,7,7,7,6,7,6
1,9,6,9,8,8,5,4,8,5,6,8,7,5,6,8,9,8,8,8,7
2,7,6,6,5,7,7,4,6,5,3,6,8,10,10,8,8,10,9,10,10
3,6,6,8,10,9,11,9,5,6,6,6,5,5,5,6,6,6,6,6,7
4,5,7,7,9,7,7,8,10,8,11,8,9,9,9,9,7,8,11,9,7
5,8,9,8,10,9,9,11,8,6,7,8,8,7,8,6,6,6,8,9,8
6,4,10,11,11,9,10,12,12,7,5,6,8,10,7,10,10,12,10,9,9
7,7,11,11,11,10,8,7,9,8,8,10,11,10,12,14,13,16,13,13,14
8,8,8,10,9,9,11,12,10,9,9,8,10,10,12,14,10,11,13,11,10
9,10,8,9,9,7,6,6,9,8,9,11,12,12,9,9,15,12,13,13,13


### opt

In [33]:
for dir in dirs_opt_temp1:
    comparative_stats(dir, base_opt_sentimap_temp1)

showing result for directory gemini_2pos_opt_senti_hpt


,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19
0,5,3,3,3,3,3,3,4,3,4,5,6,6,7,6,5,4,4,4,3
1,5,6,6,4,4,5,4,5,2,2,3,4,6,7,5,7,3,3,2,3
2,4,5,10,5,3,2,2,2,2,2,1,2,2,2,2,2,2,2,1,2
3,4,2,4,2,2,2,2,2,2,2,1,0,1,2,1,2,2,2,2,2
4,4,6,3,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2
5,4,5,3,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2
6,4,6,3,3,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2
7,5,4,3,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2
8,6,4,4,2,1,2,2,2,1,2,2,2,2,2,2,2,2,2,1,1
9,4,4,5,2,2,2,0,1,1,1,2,2,1,0,1,1,1,1,1,1


showing result for directory gemini_2neg_opt_senti_hpt


,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19
0,5,6,8,6,8,7,7,6,8,6,7,7,8,5,5,7,6,5,6,5
1,4,6,6,8,7,6,7,9,7,4,4,4,6,6,6,6,4,5,6,6
2,4,8,5,9,6,8,6,7,6,6,6,6,6,6,6,6,6,6,6,5
3,5,3,6,5,7,5,7,9,11,7,7,6,6,6,6,6,6,6,6,6
4,5,4,6,6,6,7,8,6,6,5,5,6,6,6,6,6,5,6,6,6
5,5,4,4,4,3,5,7,9,8,6,6,7,6,5,6,4,4,5,4,3
6,5,7,6,5,5,6,6,7,7,8,5,6,6,6,6,6,6,6,6,6
7,4,5,6,6,8,7,8,8,7,7,6,6,7,7,6,6,6,6,6,6
8,3,6,4,7,5,9,6,7,7,6,7,6,6,6,6,6,6,6,6,7
9,3,5,3,7,4,7,6,8,7,5,5,6,6,6,6,6,6,7,7,6


showing result for directory gemini_sent_2pos_opt_senti_hpt


,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19
0,6,6,2,3,5,6,6,6,6,5,5,5,5,5,6,6,5,5,5,4
1,6,6,5,7,5,7,7,7,10,8,7,5,7,3,5,7,6,7,8,6
2,6,5,4,5,4,3,3,4,1,4,4,2,2,2,2,3,2,2,2,2
3,4,4,4,3,2,2,1,1,2,1,1,2,2,2,2,2,2,2,2,2
4,6,6,5,4,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2
5,4,6,4,5,3,3,4,2,1,3,2,2,2,2,2,2,2,2,2,2
6,3,6,4,5,2,3,2,2,2,2,2,1,1,2,2,2,2,3,3,2
7,4,5,5,2,2,1,1,1,0,1,1,1,2,1,2,1,1,1,1,1
8,4,6,4,1,3,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2
9,5,6,3,3,2,2,2,2,2,2,2,2,2,2,2,1,2,2,2,2


showing result for directory gemini_sent_2neg_opt_senti_hpt


,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19
0,3,4,6,6,7,6,7,5,4,5,7,5,5,6,6,6,4,4,4,5
1,5,4,5,7,6,7,5,5,3,4,5,7,7,7,6,6,5,5,6,7
2,4,5,7,9,9,6,7,7,6,8,9,5,7,6,8,6,3,9,6,8
3,6,7,8,8,6,6,6,7,6,7,6,7,6,6,6,7,6,6,7,7
4,6,6,6,4,5,6,4,4,6,5,5,5,5,6,6,6,7,6,7,7
5,5,6,7,7,5,6,6,7,6,7,6,8,7,7,9,8,8,7,8,9
6,5,5,6,6,5,7,7,9,7,7,5,6,6,6,6,6,7,6,6,6
7,5,6,5,5,5,7,6,6,5,6,7,6,8,7,6,6,6,6,6,6
8,5,5,6,6,6,6,8,6,6,6,5,6,6,6,6,6,6,6,6,6
9,6,7,8,7,6,5,7,7,8,6,9,7,8,8,6,8,7,7,7,7
